# Lekcja 37: Redukcja wymiarowości — teoria i przykłady (do druku)

**Tagi:** `#pca` `#tsne` `#umap` `#redukcja-wymiarów` `#wizualizacja`

W tej lekcji poznasz techniki redukcji wymiarowości – metody, które zmniejszają liczbę cech (wymiarów) danych przy zachowaniu jak najwięcej informacji. Zaczniemy od **PCA** (Principal Component Analysis) – klasycznej metody liniowej opartej na algebrze liniowej. Następnie przejdziemy do **t-SNE** i **UMAP** – nieliniowych metod idealnych do wizualizacji danych wysokowymiarowych.

> Każdy przykład jest w **osobnej komórce Python**. Na końcu: zadania 1–20 (bez rozwiązań).
> Wydruk: File → Print / Export PDF.


## 1. Przekleństwo wymiarowości

**Definicja.** Przekleństwo wymiarowości (*curse of dimensionality*) to zjawisko, w którym wraz ze wzrostem liczby wymiarów (cech) dane stają się coraz bardziej „rzadkie” – odległości między punktami stają się podobne, klasteryzacja i klasyfikacja stają się trudniejsze, a modele wymagają wykładniczo więcej danych do skutecznego uczenia.

| Problem | Opis | Efekt |
|---------|------|-------|
| Odległości się „spłaszczają” | W 1000D różnica między min a max odległością jest minimalna | kNN i k-Means przestają działać |
| Overfitting | Więcej cech niż próbek → model „zapamiętuje” szum | Niska generalizacja |
| Czas obliczeń | Wiele algorytmów ma złożoność O(d²) lub O(d³) | Wolne trening i predykcja |
| Korelacje między cechami | Redundantne cechy nie dodają informacji | Marnowanie zasobów |

> **Notatka.** Reguła kciuka: jeśli `d > n`, **MUSISZ** zredukować wymiary lub użyć regularyzacji. W praktyce nawet `d > 50` może być problematyczne.


## 2. PCA – Principal Component Analysis

**Definicja.** PCA znajduje nowe osie (główne składowe), wzdłuż których dane mają **największą wariancję**. PC1 wyjaśnia najwięcej, PC2 – najwięcej z reszty itd. Osie są ortogonalne.

**Kroki:** 1) standaryzacja → 2) macierz kowariancji → 3) eigenvalues/eigenvectors → 4) sortowanie → 5) top-k → 6) `X_new = X @ V_k`

> **Tip.** Ile składowych? Scree Plot + próg 95% albo `PCA(n_components=0.95)`.

### Przykład 1: PCA od zera w NumPy (Iris)


In [ ]:
# PCA od zera -- implementacja krok po kroku
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

print(f"Oryginalne dane: {X.shape} (150 próbek, 4 cechy)")

# === Krok 1: Standaryzacja (mean=0, std=1) ===
scaler = StandardScaler()
X_std = scaler.fit_transform(X)
print(f"Średnie po standaryzacji: {X_std.mean(axis=0).round(10)}")

# === Krok 2: Macierz kowariancji ===
cov_matrix = np.cov(X_std, rowvar=False)
print("\nMacierz kowariancji (4x4):")
print(pd.DataFrame(cov_matrix.round(3), columns=feature_names, index=feature_names))

# === Krok 3: Eigenvalues i Eigenvectors ===
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
sorted_idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_idx]
eigenvectors = eigenvectors[:, sorted_idx]

explained_var = eigenvalues / eigenvalues.sum()
print(f"\nEigenvalues: {eigenvalues.round(4)}")
print("Wariancja wyjaśniona (%):")
for i, (ev, var) in enumerate(zip(eigenvalues, explained_var)):
    print(
        f"  PC{i+1}: eigenvalue={ev:.4f}, wariancja={var:.1%}, "
        f"kumulatywna={explained_var[:i+1].sum():.1%}"
    )

# === Krok 4: Transformacja (projekcja na 2 składowe) ===
n_components = 2
V_k = eigenvectors[:, :n_components]
X_pca = X_std @ V_k

print(f"\nDane po PCA: {X_pca.shape} (150 próbek, 2 składowe)")
print(f"Zachowana wariancja: {explained_var[:n_components].sum():.1%}")

# === Wizualizacja ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#ff6b6b", "#4ecdc4", "#ffd93d"]
for i, (name, color) in enumerate(zip(iris.target_names, colors)):
    mask = y == i
    axes[0].scatter(
        X_pca[mask, 0], X_pca[mask, 1], c=color, label=name,
        alpha=0.7, s=50, edgecolors="gray", linewidth=0.5,
    )
axes[0].set_xlabel(f"PC1 ({explained_var[0]:.1%} wariancji)", fontsize=12)
axes[0].set_ylabel(f"PC2 ({explained_var[1]:.1%} wariancji)", fontsize=12)
axes[0].set_title("PCA: Iris w 2D", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(1, 5), explained_var, alpha=0.7, color="#4ecdc4", label="Pojedyncza")
axes[1].plot(range(1, 5), np.cumsum(explained_var), "ro-", linewidth=2, label="Kumulatywna")
axes[1].axhline(y=0.95, color="gray", linestyle="--", alpha=0.5, label="95% próg")
axes[1].set_xlabel("Składowa główna (PC)", fontsize=12)
axes[1].set_ylabel("Wariancja wyjaśniona", fontsize=12)
axes[1].set_title("Scree Plot: ile składowych zachować?", fontsize=13, fontweight="bold")
axes[1].set_xticks(range(1, 5))
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Przykład 2: PCA z sklearn na danych wysokowymiarowych (digits)


In [ ]:
# PCA na danych MNIST (cyfry 0-9, 64 piksele --> 2D)
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

digits = load_digits()
X = digits.data
y = digits.target

print(f"Dane: {X.shape} (1797 obrazów, 64 piksele)")
print(f"Klasy: {np.unique(y)} (cyfry 0-9)")

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# === PCA z zachowaniem 95% wariancji ===
pca_95 = PCA(n_components=0.95, random_state=42)
X_95 = pca_95.fit_transform(X_std)
print(
    f"\nPCA 95%: {X.shape[1]} --> {X_95.shape[1]} wymiarów "
    f"(redukcja {1 - X_95.shape[1]/X.shape[1]:.0%})"
)

# === PCA do wizualizacji (2D) ===
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_std)
print(f"PCA 2D: {pca_2d.explained_variance_ratio_.sum():.1%} wariancji zachowanej")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_map = plt.cm.tab10(np.linspace(0, 1, 10))

for digit in range(10):
    mask = y == digit
    axes[0].scatter(
        X_2d[mask, 0], X_2d[mask, 1], c=[colors_map[digit]],
        label=str(digit), alpha=0.5, s=15,
    )
axes[0].set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})", fontsize=11)
axes[0].set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})", fontsize=11)
axes[0].set_title("PCA 2D: MNIST Digits", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=8, ncol=5, loc="upper right")
axes[0].grid(True, alpha=0.3)

pca_full = PCA(random_state=42)
pca_full.fit(X_std)
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = int(np.argmax(cumsum >= 0.95) + 1)

axes[1].plot(range(1, len(cumsum) + 1), cumsum, "b-", linewidth=2)
axes[1].axhline(y=0.95, color="red", linestyle="--", label="95% próg")
axes[1].axvline(x=n_95, color="green", linestyle="--", label=f"n={n_95} składowych")
axes[1].fill_between(range(1, n_95 + 1), cumsum[:n_95], alpha=0.2, color="green")
axes[1].set_xlabel("Liczba składowych", fontsize=11)
axes[1].set_ylabel("Kumulatywna wariancja", fontsize=11)
axes[1].set_title(f"Scree Plot: {n_95} składowych dla 95%", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Ładunki PC1 (top 5 pikseli) ===")
loadings_pc1 = pca_2d.components_[0]
top_features = np.argsort(np.abs(loadings_pc1))[::-1][:5]
for idx in top_features:
    print(f"  Piksel {idx} (wiersz {idx//8}, kolumna {idx%8}): {loadings_pc1[idx]:+.4f}")


### Przykład 3: PCA jako preprocessing dla ML

> **Warning.** PCA jest **liniowe**. Przy nieliniowej strukturze (spirale, moons) użyj t-SNE lub UMAP.


In [ ]:
# PCA jako krok preprocessingu -- przyspieszenie treningu ML
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.datasets import load_digits
import time

digits = load_digits()
X_train, y_train = digits.data, digits.target

configs = {
    "Bez PCA (64D)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "PCA 95%": make_pipeline(
        StandardScaler(), PCA(n_components=0.95, random_state=42), LogisticRegression(max_iter=5000)
    ),
    "PCA 20D": make_pipeline(
        StandardScaler(), PCA(n_components=20, random_state=42), LogisticRegression(max_iter=5000)
    ),
    "PCA 10D": make_pipeline(
        StandardScaler(), PCA(n_components=10, random_state=42), LogisticRegression(max_iter=5000)
    ),
    "PCA 2D": make_pipeline(
        StandardScaler(), PCA(n_components=2, random_state=42), LogisticRegression(max_iter=5000)
    ),
}

print("=== PCA jako preprocessing dla Logistic Regression ===\n")
print(f"{'Konfiguracja':<20s} | {'Accuracy':>10s} | {'Czas (s)':>10s}")
print("-" * 50)

for name, pipe in configs.items():
    start = time.perf_counter()
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    elapsed = time.perf_counter() - start
    print(f"{name:<20s} | {scores.mean():>9.4f} | {elapsed:>9.2f}")


## 3. t-SNE – wizualizacja danych wysokowymiarowych

**Definicja.** t-SNE to nieliniowa redukcja pod wizualizację 2D/3D. Zachowuje **lokalne** relacje.

| Cecha | PCA | t-SNE | UMAP |
|-------|-----|-------|------|
| Typ | Liniowy | Nieliniowy | Nieliniowy |
| Zachowuje | Globalną strukturę | Lokalną strukturę | Lokalną + częściowo globalną |
| Nowe dane | `transform()` | **NIE** | `transform()` |
| Zastosowanie | Preprocessing + wizualizacja | **Tylko** wizualizacja | Wizualizacja + preprocessing |

> **Warning.** t-SNE **NIE** do preprocessingu ML — brak `transform()`.

### Przykład 4: t-SNE na MNIST


In [ ]:
# t-SNE -- wizualizacja MNIST w 2D
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
import numpy as np
import matplotlib.pyplot as plt
import time

digits = load_digits()
y = digits.target
X_std = StandardScaler().fit_transform(digits.data)
colors_map = plt.cm.tab10(np.linspace(0, 1, 10))

print("Obliczanie t-SNE (może potrwać)...")
start = time.perf_counter()
tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    max_iter=1000,
    random_state=42,
    init="pca",
)
X_tsne = tsne.fit_transform(X_std)
tsne_time = time.perf_counter() - start
print(f"t-SNE czas: {tsne_time:.1f}s")
print(f"KL divergence: {tsne.kl_divergence_:.4f}")

pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_std)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, X_embed, title in [(axes[0], X_pca, "PCA"), (axes[1], X_tsne, "t-SNE")]:
    for digit in range(10):
        mask = y == digit
        ax.scatter(
            X_embed[mask, 0], X_embed[mask, 1], c=[colors_map[digit]],
            label=str(digit), alpha=0.6, s=15,
        )
    ax.set_title(f"{title}: MNIST Digits", fontsize=13, fontweight="bold")
    ax.legend(fontsize=8, ncol=5, loc="upper right")
    ax.grid(True, alpha=0.3)

plt.suptitle("PCA vs t-SNE: wizualizacja 64D danych w 2D", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


> **Success.** t-SNE zwykle daje czytelniejsze klastry niż PCA na nieliniowych danych.

### Przykład 5: Wpływ perplexity na t-SNE

- Niska (5): ścisłe klastry; wysoka (50+): rozmyte
- Start: `perplexity=30`, `init="pca"`, zawsze `random_state`
- Duże dane (>10k): najpierw PCA → 50D, potem t-SNE


In [ ]:
# Perplexity -- kluczowy hiperparametr t-SNE
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt
import numpy as np

digits = load_digits()
y = digits.target
X_std = StandardScaler().fit_transform(digits.data)
colors_map = plt.cm.tab10(np.linspace(0, 1, 10))

perplexities = [5, 10, 30, 50, 100]
fig, axes = plt.subplots(1, len(perplexities), figsize=(20, 4))

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42, max_iter=1000, init="pca")
    X_embed = tsne.fit_transform(X_std)
    for digit in range(10):
        mask = y == digit
        ax.scatter(X_embed[mask, 0], X_embed[mask, 1], c=[colors_map[digit]], alpha=0.5, s=8)
    ax.set_title(f"perplexity={perp}", fontsize=11, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("t-SNE: wpływ perplexity na wizualizację", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. UMAP – nowoczesna alternatywa

**Definicja.** UMAP łączy zalety PCA i t-SNE: lokalna + globalna struktura, szybszy niż t-SNE, **ma** `transform()`.

**Parametry:** `n_neighbors` (lokalna vs globalna), `min_dist` (ścisłość klastrów), `metric`, `n_components`.

### Przykład 6: UMAP vs t-SNE vs PCA


In [ ]:
# UMAP -- porównanie z PCA i t-SNE
%pip install -q umap-learn

import umap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import time

digits = load_digits()
y = digits.target
colors_map = plt.cm.tab10(np.linspace(0, 1, 10))
X_std = StandardScaler().fit_transform(digits.data)

methods = {}

start = time.perf_counter()
pca = PCA(n_components=2, random_state=42)
methods["PCA"] = {"embed": pca.fit_transform(X_std), "time": time.perf_counter() - start}

start = time.perf_counter()
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca")
methods["t-SNE"] = {"embed": tsne.fit_transform(X_std), "time": time.perf_counter() - start}

start = time.perf_counter()
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
methods["UMAP"] = {"embed": reducer.fit_transform(X_std), "time": time.perf_counter() - start}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, data) in zip(axes, methods.items()):
    X_embed = data["embed"]
    for digit in range(10):
        mask = y == digit
        ax.scatter(
            X_embed[mask, 0], X_embed[mask, 1], c=[colors_map[digit]],
            label=str(digit), alpha=0.5, s=12,
        )
    ax.set_title(f"{name} ({data['time']:.1f}s)", fontsize=13, fontweight="bold")
    ax.legend(fontsize=7, ncol=5, loc="upper right")
    ax.grid(True, alpha=0.3)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("PCA vs t-SNE vs UMAP: wizualizacja MNIST", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n=== Czasy obliczeń ===")
for name, data in methods.items():
    print(f"  {name:8s}: {data['time']:.2f}s")


### Przykład 6b: kluczowy wpływ parametrów UMAP (`n_neighbors`, `min_dist`)


In [ ]:
# UMAP: wpływ n_neighbors i min_dist
import umap
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

digits = load_digits()
y = digits.target
X_std = StandardScaler().fit_transform(digits.data)
colors_map = plt.cm.tab10(np.linspace(0, 1, 10))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, nn in zip(axes[0], [5, 15, 50]):
    reducer = umap.UMAP(n_components=2, n_neighbors=nn, min_dist=0.1, random_state=42)
    X_embed = reducer.fit_transform(X_std)
    for digit in range(10):
        mask = y == digit
        ax.scatter(X_embed[mask, 0], X_embed[mask, 1], c=[colors_map[digit]], alpha=0.5, s=8)
    ax.set_title(f"n_neighbors={nn}", fontsize=12, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])

for ax, md in zip(axes[1], [0.0, 0.1, 0.5]):
    reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=md, random_state=42)
    X_embed = reducer.fit_transform(X_std)
    for digit in range(10):
        mask = y == digit
        ax.scatter(X_embed[mask, 0], X_embed[mask, 1], c=[colors_map[digit]], alpha=0.5, s=8)
    ax.set_title(f"min_dist={md}", fontsize=12, fontweight="bold")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("UMAP: wpływ n_neighbors i min_dist", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Zastosowania praktyczne

### Przykład 7: PCA + k-Means + biplot (California Housing)

> **Tip.** Biplot: strzałki cech = kierunek i siła wpływu na PC1/PC2.


In [ ]:
# Pipeline: PCA do redukcji + k-Means do klasteryzacji
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import fetch_california_housing
import matplotlib.pyplot as plt

housing = fetch_california_housing()
X = housing.data
feature_names = housing.feature_names

print(f"Dane: {X.shape} (20640 domów, 8 cech)")
print(f"Cechy: {feature_names}")

X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"\nPCA: {pca.explained_variance_ratio_.sum():.1%} wariancji w 2D")
print(f"Ładunki PC1: {dict(zip(feature_names, pca.components_[0].round(3)))}")
print(f"Ładunki PC2: {dict(zip(feature_names, pca.components_[1].round(3)))}")

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)
sil = silhouette_score(X_pca, labels)
print(f"\nk-Means (k=4): Silhouette = {sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#ff6b6b", "#4ecdc4", "#ffd93d", "#a855f7"]

for i in range(4):
    mask = labels == i
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[i], alpha=0.3, s=5, label=f"Klaster {i}")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", fontsize=11)
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", fontsize=11)
axes[0].set_title("PCA + k-Means: California Housing", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

ax = axes[1]
for i in range(4):
    mask = labels == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[i], alpha=0.2, s=3)

scale = 3
for i, name in enumerate(feature_names):
    ax.arrow(
        0, 0, pca.components_[0, i] * scale, pca.components_[1, i] * scale,
        head_width=0.1, head_length=0.05, fc="black", ec="black",
    )
    ax.text(
        pca.components_[0, i] * scale * 1.15,
        pca.components_[1, i] * scale * 1.15,
        name, fontsize=9, fontweight="bold", ha="center",
    )

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})", fontsize=11)
ax.set_title("Biplot: kierunki cech w PCA", fontsize=13, fontweight="bold")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Przykład 8: Wizualizacja embeddingów NLP z t-SNE


In [ ]:
# Wizualizacja word embeddings (symulacja 50D) za pomocą t-SNE
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

np.random.seed(42)
categories = {
    "Zwierzęta": ["kot", "pies", "konik", "rybka", "ptak", "krolik", "mysz", "zolw", "ryba", "waz"],
    "Kolory": ["czerwony", "niebieski", "zielony", "zolty", "bialy", "czarny", "rozowy", "fiolet", "brazowy", "szary"],
    "Jedzenie": ["chleb", "ser", "mleko", "jablko", "banan", "ryba2", "mieso", "ryz", "makaron", "zupa"],
    "Kraje": ["polska", "niemcy", "francja", "wlochy", "hiszpania", "anglia", "japonia", "chiny", "usa", "brazylia"],
}

words, embeddings, labels = [], [], []
for cat_idx, (_, cat_words) in enumerate(categories.items()):
    for word in cat_words:
        words.append(word)
        labels.append(cat_idx)
        centroid = np.zeros(50)
        centroid[cat_idx * 10:(cat_idx + 1) * 10] = np.random.randn(10) * 2
        embeddings.append(centroid + np.random.randn(50) * 0.5)

X_emb = np.array(embeddings)
y_emb = np.array(labels)

tsne = TSNE(n_components=2, perplexity=8, random_state=42, max_iter=2000)
X_tsne = tsne.fit_transform(X_emb)

fig, ax = plt.subplots(figsize=(12, 9))
cat_names = list(categories.keys())
cat_colors = ["#ff6b6b", "#4ecdc4", "#ffd93d", "#a855f7"]

for cat_idx, (cat_name, color) in enumerate(zip(cat_names, cat_colors)):
    mask = y_emb == cat_idx
    ax.scatter(
        X_tsne[mask, 0], X_tsne[mask, 1], c=color, s=100,
        alpha=0.8, label=cat_name, edgecolors="gray", linewidth=0.5,
    )
    for i in np.where(mask)[0]:
        ax.annotate(
            words[i], (X_tsne[i, 0], X_tsne[i, 1]),
            fontsize=8, fontweight="bold", xytext=(5, 5), textcoords="offset points",
        )

ax.set_title("t-SNE: wizualizacja word embeddings (50D --> 2D)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()


### Przykład 9: Porównanie metod na różnych kształtach danych


In [ ]:
# Porównanie PCA, t-SNE, UMAP na Swiss Roll / Moons / Circles
from sklearn.datasets import make_swiss_roll, make_moons, make_circles
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

X_swiss, y_swiss = make_swiss_roll(n_samples=1000, noise=0.5, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)
X_moons = np.hstack([X_moons, rng.normal(0, 0.05, size=(300, 8))])
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)
X_circles = np.hstack([X_circles, rng.normal(0, 0.05, size=(300, 8))])

datasets = {
    "Swiss Roll (3D)": (X_swiss, y_swiss),
    "Moons (10D)": (X_moons, y_moons),
    "Circles (10D)": (X_circles, y_circles),
}

methods_fn = {
    "PCA": lambda X: PCA(n_components=2, random_state=42).fit_transform(X),
    "t-SNE": lambda X: TSNE(
        n_components=2, perplexity=30, random_state=42, init="pca"
    ).fit_transform(X),
    "UMAP": lambda X: umap.UMAP(n_components=2, random_state=42).fit_transform(X),
}

fig, axes = plt.subplots(3, 3, figsize=(15, 13))

for row, (ds_name, (X, y)) in enumerate(datasets.items()):
    # Centrujemy dane, ale nie skalujemy każdej cechy szumu do wariancji 1,
    # żeby 8 małych losowych cech nie zagłuszało właściwego kształtu.
    X_centered = X - X.mean(axis=0)
    for col, (method_name, method_fn) in enumerate(methods_fn.items()):
        X_2d = method_fn(X_centered)
        ax = axes[row, col]
        ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap="coolwarm", s=10, alpha=0.7)
        ax.set_title(method_name if row == 0 else "", fontsize=12, fontweight="bold")
        if col == 0:
            ax.set_ylabel(ds_name, fontsize=11, fontweight="bold")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(True, alpha=0.2)

plt.suptitle("PCA vs t-SNE vs UMAP na różnych kształtach danych", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Kiedy użyć której metody?

| Scenariusz | Rekomendacja | Dlaczego |
|------------|--------------|----------|
| EDA, małe dane | **t-SNE** | Najlepsza separacja klastrów |
| EDA, duże dane (>10k) | **UMAP** | 10–100× szybszy niż t-SNE |
| Preprocessing przed ML | **PCA** | Interpretowalny, `transform()`, szybki |
| Preprocessing nieliniowy | **UMAP** | Ma `transform()` |
| Kompresja / szum | **PCA** | Małe eigenvalues = szum |
| Interpretowalność cech | **PCA** | Ładunki (loadings) |

## 7. Aspekt środowiskowy (Green IT)

- PCA jako preprocessing przyspiesza ML
- Duże dane: PCA → 50D, potem t-SNE; albo od razu UMAP
- Do EDA wystarczy 10–20% próbek
- `float32`, `IncrementalPCA` przy dużych zbiorach

---

# Zadania (bez rozwiązań)

Poniżej polecenia z PDF. Każde zadanie ma pustą komórkę Python do samodzielnego rozwiązania.


## Zadania proste (1–8)


### ✏ Zadanie 1 – PCA ręcznie

Dla 5 punktów 2D: `(1,2), (2,4), (3,6), (4,8), (5,10)`:

- Oblicz macierz kowariancji
- Znajdź eigenvalues i eigenvectors
- Wykonaj PCA do 1D
- Ile wariancji zachowano?

**Oczekiwany wynik:** obliczenia krok po kroku

**Poziom:** proste


In [ ]:
# Zadanie 1 – PCA ręcznie
# Twój kod tutaj


### ✏ Zadanie 2 – Scree Plot na Iris

Dla datasetu Iris:

- Zastosuj PCA (pełne, 4 składowe)
- Narysuj Scree Plot (wariancja wyjaśniona i kumulatywna)
- Ile składowych potrzeba dla 90%, 95%, 99%?

**Oczekiwany wynik:** Scree Plot z progami

**Poziom:** proste


In [ ]:
# Zadanie 2 – Scree Plot na Iris
# Twój kod tutaj


### ✏ Zadanie 3 – PCA na Wine

Dla datasetu Wine (13 cech):

- Zastosuj PCA do 2D i 3D
- Zwizualizuj z kolorami klas
- Pokaż ładunki (loadings) – które cechy dominują w PC1 i PC2?

**Oczekiwany wynik:** wykresy 2D i 3D, tabela ładunków

**Poziom:** proste


In [ ]:
# Zadanie 3 – PCA na Wine
# Twój kod tutaj


### ✏ Zadanie 4 – t-SNE na Iris

Zastosuj t-SNE na Iris i porównaj z PCA. Przetestuj `perplexity = [5, 15, 30, 50]`. Który daje najczytelniejsze klastry?

**Oczekiwany wynik:** siatka wizualizacji PCA vs t-SNE z różnymi perplexity

**Poziom:** proste


In [ ]:
# Zadanie 4 – t-SNE na Iris
# Twój kod tutaj


### ✏ Zadanie 5 – Rekonstrukcja z PCA

Dla MNIST digits:

- Zastosuj PCA z `n_components = [2, 5, 10, 20, 40, 64]`
- Dokonaj rekonstrukcji (`inverse_transform`)
- Zwizualizuj odtworzone obrazy – przy ilu składowych obraz jest rozpoznawalny?

**Oczekiwany wynik:** siatka oryginalnych i odtworzonych obrazów

**Poziom:** proste


In [ ]:
# Zadanie 5 – Rekonstrukcja z PCA
# Twój kod tutaj


### ✏ Zadanie 6 – PCA jako preprocessing

Porównaj accuracy RandomForest na MNIST digits:

- Bez PCA (64 cechy)
- PCA 95% wariancji
- PCA 20 składowych
- PCA 5 składowych

Zmierz accuracy (`cross_val_score`) i czas treningu.

**Oczekiwany wynik:** tabela accuracy vs czas vs wymiar

**Poziom:** proste


In [ ]:
# Zadanie 6 – PCA jako preprocessing
# Twój kod tutaj


### ✏ Zadanie 7 – UMAP podstawy

Zastosuj UMAP na MNIST digits z różnymi parametrami:

- `n_neighbors = [5, 15, 50]`
- `min_dist = [0.0, 0.1, 0.5]`

**Oczekiwany wynik:** siatka 3×3 wizualizacji

**Poziom:** proste


In [ ]:
# Zadanie 7 – UMAP podstawy
# Twój kod tutaj


### ✏ Zadanie 8 – Porównanie 3 metod

Na jednym datasecie (`Breast Cancer`, `load_breast_cancer`) porównaj PCA, t-SNE i UMAP:

- Wizualizacja 2D z kolorami klas
- Czas obliczeń
- Która metoda najlepiej separuje klasy?

**Oczekiwany wynik:** 3 wykresy, tabela porównawcza

**Poziom:** proste


In [ ]:
# Zadanie 8 – Porównanie 3 metod
# Twój kod tutaj


## Zadania średnie (9–12)


### ✏ Zadanie 9 – Biplot interaktywny

Dla California Housing:

- Zastosuj PCA do 2D
- Stwórz biplot z Plotly (interaktywny): punkty + strzałki cech
- Dodaj kolorowanie według ceny domu (target)

**Oczekiwany wynik:** interaktywny biplot

**Poziom:** średnie


In [ ]:
# Zadanie 9 – Biplot interaktywny
# Twój kod tutaj


### ✏ Zadanie 10 – PCA + klasteryzacja pipeline

Na datasecie 20 Newsgroups (TF-IDF wektory):

- Zastosuj PCA (50D) lub TruncatedSVD (dla sparse)
- k-Means na zredukowanych danych
- t-SNE do wizualizacji klastrów
- Porównaj klastry z prawdziwymi kategoriami

**Oczekiwany wynik:** wizualizacja klastrów z etykietami

**Poziom:** średnie


In [ ]:
# Zadanie 10 – PCA + klasteryzacja pipeline
# Twój kod tutaj


### ✏ Zadanie 11 – Kernel PCA

Zastosuj Kernel PCA (`KernelPCA` w sklearn) na danych `make_circles` i `make_moons`:

- Porównaj kernele: `"linear"`, `"rbf"`, `"poly"`
- Porównaj z zwykłym PCA
- Który kernel najlepiej radzi sobie z nieliniowymi danymi?

**Oczekiwany wynik:** porównanie PCA vs Kernel PCA

**Poziom:** średnie


In [ ]:
# Zadanie 11 – Kernel PCA
# Twój kod tutaj


### ✏ Zadanie 12 – t-SNE na embeddingach BERT

Pobierz embeddingi `[CLS]` z DistilBERT dla 500 tekstów z 20 Newsgroups (4 kategorie). Zwizualizuj za pomocą t-SNE i UMAP. Czy kategorie są wyraźnie oddzielone?

**Oczekiwany wynik:** wizualizacja z kolorami kategorii

**Poziom:** średnie


In [ ]:
# Zadanie 12 – t-SNE na embeddingach BERT
# Twój kod tutaj


## Zadania trudne – challenge (13–20)


### ✏ Zadanie 13 – PCA od zera

Zaimplementuj PCA od zera w NumPy:

- Standaryzacja
- Macierz kowariancji
- Eigendecomposition
- Wybór top-k składowych
- `fit()`, `transform()`, `inverse_transform()`

Porównaj z sklearn PCA na Iris.

**Oczekiwany wynik:** implementacja PCA, porównanie z sklearn

**Poziom:** challenge


In [ ]:
# Zadanie 13 – PCA od zera
# Twój kod tutaj


### ✏ Zadanie 14 – t-SNE od zera

Zaimplementuj uproszczoną wersję t-SNE:

- Oblicz macierz prawdopodobieństw par (rozkład Gaussa) w przestrzeni oryginalnej
- Inicjalizuj losowo w 2D
- Optymalizuj KL divergence za pomocą gradient descent
- Porównaj z sklearn TSNE

**Oczekiwany wynik:** implementacja t-SNE, wizualizacja konwergencji

**Poziom:** challenge


In [ ]:
# Zadanie 14 – t-SNE od zera
# Twój kod tutaj


### ✏ Zadanie 15 – IncrementalPCA na dużych danych

Użyj IncrementalPCA do przetwarzania dużego datasetu batchami:

- Wygeneruj dane 100k × 100D
- IncrementalPCA z `batch_size=1000`
- Porównaj z PCA (pełna macierz kowariancji)
- Zmierz czas i zużycie pamięci

**Oczekiwany wynik:** porównanie IncrementalPCA vs PCA

**Poziom:** challenge


In [ ]:
# Zadanie 15 – IncrementalPCA na dużych danych
# Twój kod tutaj


### ✏ Zadanie 16 – TruncatedSVD na sparse danych

Dla danych tekstowych (TF-IDF, sparse matrix):

- Użyj TruncatedSVD zamiast PCA (działa na sparse)
- Porównaj z NMF (Non-negative Matrix Factorization)
- Zinterpretuj składowe: jakie tematy reprezentują?

**Oczekiwany wynik:** porównanie SVD vs NMF, interpretacja tematów

**Poziom:** challenge


In [ ]:
# Zadanie 16 – TruncatedSVD na sparse danych
# Twój kod tutaj


### ✏ Zadanie 17 – Autoencoder do redukcji wymiarów

Zbuduj Autoencoder (Keras) do nieliniowej redukcji wymiarów:

- Encoder: 64 → 32 → 2 (latent space)
- Decoder: 2 → 32 → 64
- Trenuj na MNIST digits
- Porównaj wizualizacje latent space z PCA, t-SNE, UMAP

**Oczekiwany wynik:** porównanie 4 metod, analiza

**Poziom:** challenge


In [ ]:
# Zadanie 17 – Autoencoder do redukcji wymiarów
# Twój kod tutaj


### ✏ Zadanie 18 – Stabilność t-SNE

Zbadaj stabilność t-SNE:

- Uruchom t-SNE 10 razy z różnymi `random_state`
- Zmierz jak bardzo różnią się wyniki (Procrustes analysis)
- Porównaj stabilność z UMAP

**Oczekiwany wynik:** analiza stabilności, metryki

**Poziom:** challenge


In [ ]:
# Zadanie 18 – Stabilność t-SNE
# Twój kod tutaj


### ✏ Zadanie 19 – Redukcja wymiarów + klasyfikacja pipeline

Zbuduj kompletny pipeline: redukcja + klasyfikacja na Fashion MNIST:

- Warianty: PCA+LogReg, PCA+RF, UMAP+LogReg, UMAP+RF
- Porównaj accuracy, czas treningu, czas inferencji
- Dobierz optymalny `n_components` za pomocą GridSearchCV

**Oczekiwany wynik:** tabela porównawcza, optymalny pipeline

**Poziom:** challenge


In [ ]:
# Zadanie 19 – Redukcja wymiarów + klasyfikacja pipeline
# Twój kod tutaj


### ✏ Zadanie 20 – Green IT benchmark redukcji wymiarów

Benchmark efektywności:

- Metody: PCA, IncrementalPCA, TruncatedSVD, t-SNE, UMAP
- Rozmiary: 1k, 5k, 10k, 50k, 100k próbek (100D)
- Mierz: czas, pamięć, jakość (`trustworthiness` z sklearn)
- Stwórz rekomendacje: która metoda dla jakiego scenariusza?

**Oczekiwany wynik:** tabela benchmark, wykresy, rekomendacje

**Poziom:** challenge


In [ ]:
# Zadanie 20 – Green IT benchmark redukcji wymiarów
# Twój kod tutaj
